# Notebook 10: Correlated damage and loss

This notebook propagates the three validated Notebook 09 ground-motion
cases through the frozen Phase 1 damage and insurance models:

- **I0_PHASE1_INDEPENDENT**: exact Phase 1 control;
- **C1_ALDEA22_SUBDUCTION**: primary spatial-correlation case;
- **C2_GODA_ATKINSON09**: correlation-model sensitivity.

The experiment uses the same occurrence-site structural, nonstructural
drift, and nonstructural acceleration damage uniforms for all cases.
Structural and nonstructural damage are therefore recomputed from each
case's SA(0.4 s), but no exposure, fragility, repair-cost, policy, or
random-stream assumption changes.

Official loss distributions use sampled damage states. Conditional
expected loss is not substituted for sampled portfolio loss because
deductibles and other nonlinear financial terms must be applied after
the occurrence-building damage realization.

In [ ]:
from __future__ import annotations

from collections import Counter
import gzip
import hashlib
import io
import json
import math
import os
from pathlib import Path
import shutil
import sys
from typing import Any, Iterable

import numpy as np
import pandas as pd
from IPython.display import display


PIPELINE_VERSION = "notebook10_correlated_damage_loss_v1"
SCHEMA_VERSION = "notebook10_correlated_damage_loss_handoff_v1"
EXPECTED_SITES = 470
EXPECTED_OCCURRENCES = 10_630
EXPECTED_ROWS = EXPECTED_SITES * EXPECTED_OCCURRENCES
EXPECTED_PARTITIONS = 16
EXPECTED_CATALOG_YEARS = 2_000_000
EXPECTED_OCCUPIED_CATALOG_YEARS = 10_593
EXPECTED_ZERO_EVENT_YEARS = 1_989_407
EVENT_BATCH_SIZE = int(os.environ.get("NOTEBOOK10_EVENT_BATCH_SIZE", "64"))
GZIP_COMPRESSLEVEL = int(os.environ.get("NOTEBOOK10_GZIP_LEVEL", "1"))
VERIFY_INPUT_HASHES = os.environ.get("NOTEBOOK10_VERIFY_INPUT_HASHES", "1") == "1"
ROW_TOLERANCE_USD = 2.0e-6
AGGREGATE_TOLERANCE_USD = 0.01
RETURN_PERIODS = [
    50, 100, 200, 250, 500, 1_000, 2_000, 2_500, 5_000,
    10_000, 20_000, 50_000, 100_000, 200_000, 500_000,
    1_000_000, 2_000_000,
]
if EVENT_BATCH_SIZE <= 0:
    raise ValueError("NOTEBOOK10_EVENT_BATCH_SIZE must be positive.")
if not 1 <= GZIP_COMPRESSLEVEL <= 9:
    raise ValueError("NOTEBOOK10_GZIP_LEVEL must lie between 1 and 9.")

EXPECTED_HASHES = {
    "paired_ground_motion": "9640e9b524d5f0ee0d5d447c95867b867305a996646122b9fc778d2168d278ac",
    "structural_fragility": "6fa57de96d941af5d89995f530690187f0e657f83b38d31225b4ffd94fb15fc5",
    "nonstructural_fragility": "0504fa837e7d447485b30954d30063ad798d1cca7674842840f0d1b99a9d08d3",
    "structural_values": "7668b755563b70ed637e1c53d89cb015c07b651752f774e0ccffcc2db878817f",
    "nonstructural_values": "15b274cd59c3bfac96ccec10f4187341e3df11399e0e557ba56ee0aa3c5a79de",
    "policy_terms": "1af92add99ba6b326f282f4a0e333b9825aeb3d7ec9ed507994b6e0691232f54",
    "phase1_total_ground_up": "afc451e7418b65f7836d0d6174602325afcc7b29a85ef8cf46b883222c256b64",
    "phase1_insured_loss": "2081e04955efb56fa3743dcb217cf351de186ef49315e60a318dc6400afee86a",
}
EXPECTED_DAMAGE_NAMESPACES = {
    "structural": "notebook5_structural_damage_v1",
    "nsd": "notebook5_nonstructural_drift_damage_v1",
    "nsa": "notebook5_nonstructural_acceleration_damage_v1",
}
EXPECTED_PHASE1_TOTALS = {
    "i0_structural_ground_up_loss_2022_usd": 34_251_934_804.59501,
    "i0_nsd_ground_up_loss_2022_usd": 58_800_822_694.022255,
    "i0_nsa_ground_up_loss_2022_usd": 298_792_142_561.9628,
    "i0_total_ground_up_loss_2022_usd": 391_844_900_060.5801,
    "i0_deductible_absorbed_loss_2022_usd": 145_885_780_125.9232,
    "i0_policy_limit_absorbed_loss_2022_usd": 0.0,
    "i0_coinsurance_absorbed_loss_2022_usd": 0.0,
    "i0_gross_insured_loss_2022_usd": 245_959_119_934.65695,
    "i0_uninsured_loss_2022_usd": 145_885_780_125.9232,
}
EXPECTED_PHASE1_STATE_COUNTS = {
    "i0_structural_0": 3_290_326,
    "i0_structural_1": 867_776,
    "i0_structural_2": 644_196,
    "i0_structural_3": 141_613,
    "i0_structural_4": 52_189,
    "i0_nsd_0": 3_449_445,
    "i0_nsd_1": 654_132,
    "i0_nsd_2": 677_450,
    "i0_nsd_3": 131_878,
    "i0_nsd_4": 83_195,
    "i0_nsa_0": 2_366_776,
    "i0_nsa_1": 764_806,
    "i0_nsa_2": 783_016,
    "i0_nsa_3": 603_858,
    "i0_nsa_4": 477_644,
}
EXPECTED_PHASE1_ANNUAL_METRICS = {
    "ground_up_aal_2022_usd": 195_922.4500302902,
    "gross_insured_aal_2022_usd": 122_979.55996732853,
    "uninsured_aal_2022_usd": 72_942.89006296158,
    "maximum_ground_up_aep_2022_usd": 282_749_467.80267936,
    "maximum_ground_up_oep_2022_usd": 282_749_467.80267936,
    "maximum_gross_insured_aep_2022_usd": 244_325_807.3326858,
    "maximum_gross_insured_oep_2022_usd": 244_325_807.3326858,
    "maximum_uninsured_aep_2022_usd": 62_301_773.02983487,
    "maximum_uninsured_oep_2022_usd": 38_423_660.46999352,
    "positive_gross_insured_aep_years": 8_171,
    "positive_gross_insured_oep_years": 8_171,
    "positive_uninsured_aep_years": 9_901,
    "positive_uninsured_oep_years": 9_901,
}


def find_project_root() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "10_correlated_damage_and_loss.ipynb").exists():
            return candidate
        if (
            (candidate / "tools" / "correlated_damage_loss.py").exists()
            and (candidate / "09_generate_correlated_ground_motion_fields.ipynb").exists()
        ):
            return candidate
    raise FileNotFoundError(
        "Could not identify the seismic-correlation-insurance-loss repository root."
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from tools.correlated_damage_loss import (
    CASE_PREFIXES,
    COMPONENT_NAMESPACES,
    DAMAGE_STATES,
    GROUND_MOTION_REQUIRED_COLUMNS,
    PAIRED_LOSS_OUTPUT_COLUMNS,
    build_paired_damage_loss_batch,
    build_site_parameter_table,
    summarize_occurrences,
)


def sha256_file(path: Path, chunk_bytes: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while block := handle.read(chunk_bytes):
            digest.update(block)
    return digest.hexdigest()


def load_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(payload, indent=2, sort_keys=True, allow_nan=False) + "\n",
        encoding="utf-8",
    )
    temporary.replace(path)


def write_csv(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(temporary, index=False, lineterminator="\n")
    temporary.replace(path)


def write_gzip_csv_deterministic(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("wb") as raw:
        with gzip.GzipFile(
            filename="",
            mode="wb",
            fileobj=raw,
            compresslevel=GZIP_COMPRESSLEVEL,
            mtime=0,
        ) as compressed:
            with io.TextIOWrapper(compressed, encoding="utf-8", newline="") as text:
                frame.to_csv(
                    text,
                    index=False,
                    float_format="%.17g",
                    lineterminator="\n",
                )
    temporary.replace(path)


def combine_gzip_csv_files(source_paths: list[Path], destination: Path) -> None:
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    expected_header: bytes | None = None
    with temporary.open("wb") as raw_destination:
        with gzip.GzipFile(
            filename="",
            mode="wb",
            fileobj=raw_destination,
            compresslevel=GZIP_COMPRESSLEVEL,
            mtime=0,
        ) as destination_gzip:
            for source_path in source_paths:
                with gzip.open(source_path, "rb") as source:
                    header = source.readline()
                    if expected_header is None:
                        expected_header = header
                        destination_gzip.write(header)
                    elif header != expected_header:
                        raise RuntimeError(f"Inconsistent chunk header: {source_path}")
                    shutil.copyfileobj(source, destination_gzip, length=8 * 1024 * 1024)
    temporary.replace(destination)


def project_relative_path(path: Path) -> str:
    try:
        return path.resolve().relative_to(PROJECT_ROOT.resolve()).as_posix()
    except ValueError as exc:
        raise ValueError(f"Path is outside the project root: {path}") from exc


def resolve_repository_path(value: object) -> Path:
    text = str(value).strip().replace("\\", "/")
    marker = "/data/"
    if marker in text.lower():
        start = text.lower().index(marker) + 1
        return PROJECT_ROOT.joinpath(*text[start:].split("/"))
    path = Path(text).expanduser()
    if not path.is_absolute():
        path = PROJECT_ROOT / path
    return path.resolve()


def normalize_boolean_series(values: pd.Series) -> pd.Series:
    return (
        values.astype("string")
        .fillna("")
        .str.strip()
        .str.lower()
        .isin({"true", "1", "yes", "y"})
    )


def add_check(
    rows: list[dict[str, object]],
    check_id: str,
    passed: bool,
    detail: str,
    severity: str = "critical",
) -> None:
    rows.append(
        {
            "check_id": check_id,
            "severity": severity,
            "passed": bool(passed),
            "detail": detail,
        }
    )


def empirical_pml(losses: np.ndarray, return_period: int) -> tuple[float, int]:
    values = np.asarray(losses, dtype=np.float64)
    rank = int(math.ceil(len(values) / return_period))
    ordered = np.sort(values)[::-1]
    return float(ordered[rank - 1]), rank


NOTEBOOK9_METADATA_DIR = (
    PROJECT_ROOT / "data" / "metadata" / "phase_2" / "notebook_9_correlated_ground_motion"
)
NOTEBOOK10_METADATA_DIR = (
    PROJECT_ROOT / "data" / "metadata" / "phase_2" / "notebook_10_correlated_damage_loss"
)
NOTEBOOK10_WORK_DIR = (
    PROJECT_ROOT / "data" / "metadata" / "phase_2" / "notebook_10_correlated_damage_loss_work"
)
OUTPUT_DIR = (
    PROJECT_ROOT / "data" / "processed" / "phase_2" / "notebook_10_correlated_damage_loss"
)
CHUNK_DIR = NOTEBOOK10_WORK_DIR / "loss_chunks"
EVENT_CHUNK_DIR = NOTEBOOK10_WORK_DIR / "event_chunks"
MARKER_DIR = NOTEBOOK10_WORK_DIR / "completion_markers"
VALIDATION_DIR = NOTEBOOK10_WORK_DIR / "chunk_validation"
for directory in [
    NOTEBOOK10_METADATA_DIR,
    NOTEBOOK10_WORK_DIR,
    OUTPUT_DIR,
    CHUNK_DIR,
    EVENT_CHUNK_DIR,
    MARKER_DIR,
    VALIDATION_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

NOTEBOOK9_HANDOFF_PATH = NOTEBOOK9_METADATA_DIR / "notebook_9_final_handoff.json"
NOTEBOOK9_VALIDATION_PATH = NOTEBOOK9_METADATA_DIR / "notebook_9_final_validation.csv"
NOTEBOOK9_MANIFEST_PATH = NOTEBOOK9_METADATA_DIR / "notebook_9_chunk_manifest.csv"
SITE_ORDER_PATH = PROJECT_ROOT / "data" / "metadata" / "notebook_4_cell_18_site_order.csv"
PARAMETER_DIR = PROJECT_ROOT / "data" / "processed" / "notebook_5_damage_loss_parameters"
STRUCTURAL_FRAGILITY_PATH = PARAMETER_DIR / "seaside_w2_building_fragility_assignments.csv"
NONSTRUCTURAL_FRAGILITY_PATH = PARAMETER_DIR / "seaside_w2_building_nonstructural_fragility_assignments.csv"
STRUCTURAL_VALUE_PATH = PARAMETER_DIR / "seaside_w2_replacement_values_and_structural_ratios.csv"
NONSTRUCTURAL_VALUE_PATH = PARAMETER_DIR / "seaside_w2_replacement_values_and_nonstructural_ratios.csv"
POLICY_PATH = (
    PROJECT_ROOT / "data" / "processed" / "notebook_6_insurance_parameters"
    / "seaside_w2_baseline_policy_terms.csv"
)
PHASE1_GROUND_UP_PATH = (
    PROJECT_ROOT / "data" / "processed" / "notebook_5_total_ground_up_loss"
    / "full_total_ground_up_loss.csv.gz"
)
PHASE1_INSURED_PATH = (
    PROJECT_ROOT / "data" / "processed" / "notebook_6_baseline_insured_loss"
    / "full_baseline_gross_insured_loss.csv.gz"
)
MODULE_PATH = PROJECT_ROOT / "tools" / "correlated_damage_loss.py"

FINAL_BUILDING_LOSS_PATH = OUTPUT_DIR / "paired_damage_and_policy_loss.csv.gz"
FINAL_EVENT_LOSS_PATH = OUTPUT_DIR / "paired_event_loss_summary.csv.gz"
FINAL_ANNUAL_LOSS_PATH = OUTPUT_DIR / "paired_annual_loss_series.csv.gz"
INPUT_VALIDATION_PATH = NOTEBOOK10_METADATA_DIR / "notebook_10_input_validation.csv"
CONTROL_VALIDATION_PATH = NOTEBOOK10_METADATA_DIR / "notebook_10_control_validation.csv"
MODEL_SPECIFICATION_PATH = NOTEBOOK10_METADATA_DIR / "notebook_10_model_specification.json"
CHUNK_MANIFEST_PATH = NOTEBOOK10_METADATA_DIR / "notebook_10_chunk_manifest.csv"
CASE_SUMMARY_PATH = NOTEBOOK10_METADATA_DIR / "notebook_10_case_summary.csv"
DAMAGE_STATE_SUMMARY_PATH = NOTEBOOK10_METADATA_DIR / "notebook_10_damage_state_summary.csv"
PML_TABLE_PATH = NOTEBOOK10_METADATA_DIR / "notebook_10_pml_table.csv"
FINAL_VALIDATION_PATH = NOTEBOOK10_METADATA_DIR / "notebook_10_final_validation.csv"
FINAL_HANDOFF_PATH = NOTEBOOK10_METADATA_DIR / "notebook_10_final_handoff.json"

## Paired damage and financial design

For component $c$ and damage-state threshold $k$, the frozen Phase 1
fragility equation is

$$
P(DS_c \ge k \mid Sa)=
\Phi\left(\frac{\ln(Sa)-\ln(\theta_{c,k})}{\beta_{c,k}}\right).
$$

Mutually exclusive probabilities are obtained by differencing adjacent
exceedance probabilities. One deterministic occurrence-site uniform is
used for each component. Those three uniforms are reused across I0, C1,
and C2.

Sampled component loss is replacement value multiplied by the repair
ratio for the sampled state. Total ground-up loss is

$$
L_{GU}=L_{structural}+L_{NSD}+L_{NSA}.
$$

The unchanged Phase 1 policy waterfall is

$$
L_{insured}=
\min\left[\max(L_{eligible}-D,0),\ Limit\right]\times Coinsurance.
$$

The baseline uses full coverage, a 10 percent replacement-value
deductible, a 100 percent replacement-value limit, and 100 percent
coinsurance. The limit is retained in the calculations even though it
cannot bind while physical building loss is capped at replacement value.

In [ ]:
input_validation_rows: list[dict[str, object]] = []
required_paths = [
    NOTEBOOK9_HANDOFF_PATH,
    NOTEBOOK9_VALIDATION_PATH,
    NOTEBOOK9_MANIFEST_PATH,
    SITE_ORDER_PATH,
    STRUCTURAL_FRAGILITY_PATH,
    NONSTRUCTURAL_FRAGILITY_PATH,
    STRUCTURAL_VALUE_PATH,
    NONSTRUCTURAL_VALUE_PATH,
    POLICY_PATH,
    PHASE1_GROUND_UP_PATH,
    PHASE1_INSURED_PATH,
    MODULE_PATH,
]
for path in required_paths:
    add_check(
        input_validation_rows,
        f"required_input_{path.stem}",
        path.is_file(),
        project_relative_path(path),
    )
missing_paths = [path for path in required_paths if not path.is_file()]
if missing_paths:
    write_csv(pd.DataFrame(input_validation_rows), INPUT_VALIDATION_PATH)
    raise FileNotFoundError(
        "Notebook 10 inputs are missing: "
        + ", ".join(project_relative_path(path) for path in missing_paths)
    )

notebook9_handoff = load_json(NOTEBOOK9_HANDOFF_PATH)
notebook9_validation = pd.read_csv(NOTEBOOK9_VALIDATION_PATH)
notebook9_manifest = pd.read_csv(NOTEBOOK9_MANIFEST_PATH)
notebook9_validation_passed = normalize_boolean_series(
    notebook9_validation["passed"]
)
add_check(
    input_validation_rows,
    "notebook9_handoff_complete",
    bool(notebook9_handoff.get("notebook9_complete")),
    f"notebook9_complete={notebook9_handoff.get('notebook9_complete')}",
)
add_check(
    input_validation_rows,
    "notebook9_validation_passed",
    bool(notebook9_validation_passed.all()),
    f"failed={int((~notebook9_validation_passed).sum())}",
)
paired_record = notebook9_handoff.get("paired_output", {})
add_check(
    input_validation_rows,
    "notebook9_paired_output_hash_frozen",
    str(paired_record.get("sha256")) == EXPECTED_HASHES["paired_ground_motion"],
    f"recorded={paired_record.get('sha256')}",
)
add_check(
    input_validation_rows,
    "phase1_damage_stream_namespaces_frozen",
    COMPONENT_NAMESPACES == EXPECTED_DAMAGE_NAMESPACES,
    f"namespaces={COMPONENT_NAMESPACES}",
)
frozen = notebook9_handoff.get("frozen_controls", {})
add_check(
    input_validation_rows,
    "notebook9_catalog_dimensions_frozen",
    int(frozen.get("sites", -1)) == EXPECTED_SITES
    and int(frozen.get("occurrences", -1)) == EXPECTED_OCCURRENCES
    and int(frozen.get("catalog_years", -1)) == EXPECTED_CATALOG_YEARS,
    (
        f"sites={frozen.get('sites')}; occurrences={frozen.get('occurrences')}; "
        f"years={frozen.get('catalog_years')}"
    ),
)

required_manifest_columns = {
    "chunk_number",
    "chunk_id",
    "occurrences",
    "rows",
    "output_path",
    "output_sha256",
    "status",
}
missing_manifest_columns = sorted(
    required_manifest_columns.difference(notebook9_manifest.columns)
)
add_check(
    input_validation_rows,
    "notebook9_manifest_schema_complete",
    not missing_manifest_columns,
    f"missing={missing_manifest_columns}",
)
if missing_manifest_columns:
    raise KeyError(
        f"Notebook 09 chunk manifest is missing: {missing_manifest_columns}"
    )
notebook9_manifest["chunk_id"] = (
    notebook9_manifest["chunk_id"].astype(str).str.zfill(4)
)
notebook9_manifest = notebook9_manifest.sort_values(
    "chunk_number"
).reset_index(drop=True)
add_check(
    input_validation_rows,
    "notebook9_manifest_complete",
    len(notebook9_manifest) == EXPECTED_PARTITIONS
    and int(notebook9_manifest["rows"].sum()) == EXPECTED_ROWS
    and int(notebook9_manifest["occurrences"].sum()) == EXPECTED_OCCURRENCES
    and notebook9_manifest["status"].astype(str).str.lower().eq("complete").all(),
    (
        f"partitions={len(notebook9_manifest)}; "
        f"rows={int(notebook9_manifest['rows'].sum())}; "
        f"occurrences={int(notebook9_manifest['occurrences'].sum())}"
    ),
)

parameter_paths = {
    "structural_fragility": STRUCTURAL_FRAGILITY_PATH,
    "nonstructural_fragility": NONSTRUCTURAL_FRAGILITY_PATH,
    "structural_values": STRUCTURAL_VALUE_PATH,
    "nonstructural_values": NONSTRUCTURAL_VALUE_PATH,
    "policy_terms": POLICY_PATH,
    "phase1_total_ground_up": PHASE1_GROUND_UP_PATH,
    "phase1_insured_loss": PHASE1_INSURED_PATH,
}
parameter_hashes = {name: sha256_file(path) for name, path in parameter_paths.items()}
for name, actual_hash in parameter_hashes.items():
    add_check(
        input_validation_rows,
        f"frozen_hash_{name}",
        actual_hash == EXPECTED_HASHES[name],
        f"actual={actual_hash}; expected={EXPECTED_HASHES[name]}",
    )

site_order = pd.read_csv(SITE_ORDER_PATH).sort_values("site_ordinal")
expected_site_ids = site_order["site_id"].astype(str).tolist()
add_check(
    input_validation_rows,
    "site_order_has_470_unique_sites",
    len(expected_site_ids) == EXPECTED_SITES
    and len(set(expected_site_ids)) == EXPECTED_SITES,
    f"rows={len(expected_site_ids)}; unique={len(set(expected_site_ids))}",
)

policy_terms = pd.read_csv(POLICY_PATH)
policy_terms["policy_covered"] = normalize_boolean_series(
    policy_terms["policy_covered"]
)
site_parameters = build_site_parameter_table(
    pd.read_csv(STRUCTURAL_FRAGILITY_PATH),
    pd.read_csv(NONSTRUCTURAL_FRAGILITY_PATH),
    pd.read_csv(STRUCTURAL_VALUE_PATH),
    pd.read_csv(NONSTRUCTURAL_VALUE_PATH),
    policy_terms,
    expected_site_ids=expected_site_ids,
)
replacement_total = float(
    site_parameters["building_replacement_value_2022_usd"].sum()
)
add_check(
    input_validation_rows,
    "site_parameter_table_complete",
    len(site_parameters) == EXPECTED_SITES
    and site_parameters.notna().all().all(),
    f"rows={len(site_parameters)}; missing={int(site_parameters.isna().sum().sum())}",
)
add_check(
    input_validation_rows,
    "portfolio_replacement_value_frozen",
    math.isclose(replacement_total, 384_236_604.69993526, abs_tol=1.0e-6),
    f"replacement_value={replacement_total:.8f}",
)
add_check(
    input_validation_rows,
    "baseline_policy_terms_frozen",
    bool(
        site_parameters["policy_covered"].all()
        and np.all(site_parameters["covered_loss_share"].to_numpy() == 1.0)
        and np.all(site_parameters["coinsurance_share"].to_numpy() == 1.0)
        and np.allclose(
            site_parameters["deductible_amount_2022_usd"].to_numpy(),
            0.1
            * site_parameters[
                "building_replacement_value_2022_usd"
            ].to_numpy(),
            atol=1.0e-8,
            rtol=0.0,
        )
        and np.allclose(
            site_parameters["policy_limit_amount_2022_usd"].to_numpy(),
            site_parameters[
                "building_replacement_value_2022_usd"
            ].to_numpy(),
            atol=1.0e-8,
            rtol=0.0,
        )
    ),
    "Full coverage, 10% deductible, replacement-value limit, 100% coinsurance.",
)

input_validation = pd.DataFrame(input_validation_rows)
write_csv(input_validation, INPUT_VALIDATION_PATH)
if not normalize_boolean_series(input_validation["passed"]).all():
    display(input_validation.loc[~normalize_boolean_series(input_validation["passed"])])
    raise RuntimeError("Notebook 10 input validation failed.")

model_specification = {
    "pipeline_version": PIPELINE_VERSION,
    "phase1_release": "v1.0.0",
    "phase1_commit": "be93474ce2ab78d8002d49ae861adb641ae2741d",
    "dependence_cases": list(CASE_PREFIXES),
    "damage_states": list(DAMAGE_STATES),
    "production_im": "SA(0.4 s) in g",
    "damage_equation": "P(DS>=k|Sa)=Phi((ln(Sa)-ln(theta_k))/beta_k)",
    "sample_rule": "inverse categorical CDF using one deterministic occurrence-site uniform",
    "damage_stream_namespaces": COMPONENT_NAMESPACES,
    "paired_reuse_rule": (
        "Reuse each component's occurrence-site uniform across I0, C1, and C2; "
        "recompute damage states from each case's SA(0.4 s)."
    ),
    "ground_up_equation": "structural loss + NSD loss + NSA loss",
    "policy": {
        "scenario_id": "baseline_full_coverage_10pct_deductible_v1",
        "eligible_loss": "ground_up_loss * covered_loss_share if covered, else 0",
        "gross_insured_loss": (
            "min(max(eligible_loss - deductible, 0), policy_limit) * coinsurance_share"
        ),
        "deductible_application": "per building per occurrence",
        "coverage_share": 1.0,
        "deductible_share_of_replacement": 0.1,
        "limit_share_of_replacement": 1.0,
        "coinsurance_share": 1.0,
    },
    "scope": {
        "included": [
            "structural repair",
            "drift-sensitive nonstructural repair",
            "acceleration-sensitive nonstructural repair",
        ],
        "excluded": [
            "contents",
            "business interruption",
            "demand surge",
            "claims inflation",
        ],
    },
    "input_hashes": {
        **parameter_hashes,
        "notebook9_manifest": sha256_file(NOTEBOOK9_MANIFEST_PATH),
        "notebook9_validation": sha256_file(NOTEBOOK9_VALIDATION_PATH),
        "module": sha256_file(MODULE_PATH),
    },
}
write_json(MODEL_SPECIFICATION_PATH, model_specification)
basis_hash = hashlib.sha256(
    json.dumps(model_specification, sort_keys=True).encode("utf-8")
).hexdigest()

print("=" * 78)
print("NOTEBOOK 10 CELL 1: FROZEN DAMAGE AND POLICY INPUTS VALIDATED")
print("=" * 78)
print(f"Portfolio buildings:          {len(site_parameters):,}")
print(f"Replacement value:            ${replacement_total:,.2f}")
print(f"Notebook 09 partitions:       {len(notebook9_manifest):,}")
print(f"Damage streams:               {len(COMPONENT_NAMESPACES):,}")
print(f"Input validation checks:      {len(input_validation):,}")
print("Next: exact I0 reconstruction against Phase 1 building losses.")

In [ ]:
control_validation_rows: list[dict[str, object]] = []
first_input_path = resolve_repository_path(notebook9_manifest.iloc[0]["output_path"])
control_ground_motion = pd.read_csv(
    first_input_path,
    usecols=GROUND_MOTION_REQUIRED_COLUMNS,
    nrows=EXPECTED_SITES,
)
control_output, control_diagnostics = build_paired_damage_loss_batch(
    control_ground_motion,
    site_parameters,
)
control_occurrences = control_output["occurrence_id"].nunique()
add_check(
    control_validation_rows,
    "one_complete_control_occurrence",
    len(control_output) == EXPECTED_SITES and control_occurrences == 1,
    f"rows={len(control_output)}; occurrences={control_occurrences}",
)

phase1_ground_columns = [
    "occurrence_id",
    "site_id",
    "sampled_structural_damage_state",
    "sampled_nsd_damage_state",
    "sampled_nsa_damage_state",
    "sampled_structural_ground_up_loss_2022_usd",
    "sampled_nsd_ground_up_loss_2022_usd",
    "sampled_nsa_ground_up_loss_2022_usd",
    "sampled_total_ground_up_loss_2022_usd",
]
phase1_policy_columns = [
    "occurrence_id",
    "site_id",
    "deductible_absorbed_loss_2022_usd",
    "policy_limit_absorbed_loss_2022_usd",
    "coinsurance_absorbed_loss_2022_usd",
    "gross_insured_loss_2022_usd",
    "uninsured_loss_2022_usd",
]
phase1_ground = pd.read_csv(
    PHASE1_GROUND_UP_PATH,
    usecols=phase1_ground_columns,
    nrows=EXPECTED_SITES,
)
phase1_policy = pd.read_csv(
    PHASE1_INSURED_PATH,
    usecols=phase1_policy_columns,
    nrows=EXPECTED_SITES,
)
phase1_control = phase1_ground.merge(
    phase1_policy,
    on=["occurrence_id", "site_id"],
    how="inner",
    validate="one_to_one",
)
control_compare = control_output.merge(
    phase1_control,
    on=["occurrence_id", "site_id"],
    how="inner",
    validate="one_to_one",
)
add_check(
    control_validation_rows,
    "control_keys_match_phase1",
    len(control_compare) == EXPECTED_SITES,
    f"matched={len(control_compare)}; expected={EXPECTED_SITES}",
)

state_pairs = {
    "structural": "sampled_structural_damage_state",
    "nsd": "sampled_nsd_damage_state",
    "nsa": "sampled_nsa_damage_state",
}
for component, phase1_column in state_pairs.items():
    exact = np.array_equal(
        control_compare[f"i0_{component}_damage_state"].to_numpy(dtype=int),
        control_compare[phase1_column].to_numpy(dtype=int),
    )
    add_check(
        control_validation_rows,
        f"i0_{component}_damage_states_exact",
        exact,
        f"mismatches={int((control_compare[f'i0_{component}_damage_state'] != control_compare[phase1_column]).sum())}",
    )

numeric_pairs = {
    "i0_structural_ground_up_loss_2022_usd": "sampled_structural_ground_up_loss_2022_usd",
    "i0_nsd_ground_up_loss_2022_usd": "sampled_nsd_ground_up_loss_2022_usd",
    "i0_nsa_ground_up_loss_2022_usd": "sampled_nsa_ground_up_loss_2022_usd",
    "i0_total_ground_up_loss_2022_usd": "sampled_total_ground_up_loss_2022_usd",
    "i0_deductible_absorbed_loss_2022_usd": "deductible_absorbed_loss_2022_usd",
    "i0_policy_limit_absorbed_loss_2022_usd": "policy_limit_absorbed_loss_2022_usd",
    "i0_coinsurance_absorbed_loss_2022_usd": "coinsurance_absorbed_loss_2022_usd",
    "i0_gross_insured_loss_2022_usd": "gross_insured_loss_2022_usd",
    "i0_uninsured_loss_2022_usd": "uninsured_loss_2022_usd",
}
for notebook10_column, phase1_column in numeric_pairs.items():
    error = float(
        np.max(
            np.abs(
                control_compare[notebook10_column].to_numpy(dtype=float)
                - control_compare[phase1_column].to_numpy(dtype=float)
            ),
            initial=0.0,
        )
    )
    add_check(
        control_validation_rows,
        f"control_{notebook10_column}",
        error <= ROW_TOLERANCE_USD,
        f"maximum_error={error:.6e}",
    )

add_check(
    control_validation_rows,
    "control_component_reconciliation",
    control_diagnostics.maximum_component_reconciliation_error_2022_usd
    <= ROW_TOLERANCE_USD,
    (
        "maximum_error="
        f"{control_diagnostics.maximum_component_reconciliation_error_2022_usd:.6e}"
    ),
)
add_check(
    control_validation_rows,
    "control_policy_reconciliation",
    control_diagnostics.maximum_policy_reconciliation_error_2022_usd
    <= ROW_TOLERANCE_USD,
    (
        "maximum_error="
        f"{control_diagnostics.maximum_policy_reconciliation_error_2022_usd:.6e}"
    ),
)
add_check(
    control_validation_rows,
    "correlated_control_demands_change_results",
    bool(
        np.any(
            control_output["c1_total_ground_up_loss_2022_usd"].to_numpy()
            != control_output["i0_total_ground_up_loss_2022_usd"].to_numpy()
        )
        and np.any(
            control_output["c2_total_ground_up_loss_2022_usd"].to_numpy()
            != control_output["i0_total_ground_up_loss_2022_usd"].to_numpy()
        )
    ),
    "C1 and C2 change at least one controlled building loss.",
)

control_validation = pd.DataFrame(control_validation_rows)
write_csv(control_validation, CONTROL_VALIDATION_PATH)
if not normalize_boolean_series(control_validation["passed"]).all():
    display(control_validation.loc[~normalize_boolean_series(control_validation["passed"])])
    raise RuntimeError("Notebook 10 Phase 1 controlled reconstruction failed.")

print("=" * 78)
print("NOTEBOOK 10 CELL 2: EXACT I0 DAMAGE AND POLICY CONTROL PASSED")
print("=" * 78)
print(f"Occurrence:                   {control_output['occurrence_id'].iloc[0]}")
print(f"Sites:                        {len(control_output):,}")
print(f"Validation checks:            {len(control_validation):,}")
print("I0 sampled states:            exact Phase 1 match")
print("I0 financial losses:          within row-level tolerance")
print("Next: generate or resume all 16 paired damage-and-loss partitions.")

In [ ]:
loss_total_columns = [
    f"{prefix}_{suffix}"
    for prefix in CASE_PREFIXES.values()
    for suffix in [
        "structural_ground_up_loss_2022_usd",
        "nsd_ground_up_loss_2022_usd",
        "nsa_ground_up_loss_2022_usd",
        "total_ground_up_loss_2022_usd",
        "deductible_absorbed_loss_2022_usd",
        "policy_limit_absorbed_loss_2022_usd",
        "coinsurance_absorbed_loss_2022_usd",
        "gross_insured_loss_2022_usd",
        "uninsured_loss_2022_usd",
    ]
]
output_chunk_paths: list[Path] = []
public_manifest_rows: list[dict[str, object]] = []
event_summary_frames: list[pd.DataFrame] = []

print("=" * 78)
print("NOTEBOOK 10 CELL 3: FULL-CATALOG PAIRED DAMAGE AND POLICY LOSS")
print("=" * 78)
print(f"Partitions:                   {len(notebook9_manifest):,}")
print(f"Expected occurrences:         {EXPECTED_OCCURRENCES:,}")
print(f"Expected rows:                {EXPECTED_ROWS:,}")
print(f"Events per calculation batch: {EVENT_BATCH_SIZE:,}")
print()

for manifest_index, manifest_row in notebook9_manifest.iterrows():
    chunk_id = str(manifest_row["chunk_id"]).zfill(4)
    input_path = resolve_repository_path(manifest_row["output_path"])
    expected_input_hash = str(manifest_row["output_sha256"])
    output_path = CHUNK_DIR / f"paired_damage_loss_chunk_{chunk_id}.csv.gz"
    event_path = EVENT_CHUNK_DIR / f"paired_event_loss_chunk_{chunk_id}.csv.gz"
    marker_path = MARKER_DIR / f"paired_damage_loss_chunk_{chunk_id}.complete.json"
    validation_path = VALIDATION_DIR / f"paired_damage_loss_chunk_{chunk_id}_validation.csv"

    if not input_path.is_file():
        raise FileNotFoundError(
            f"Missing Notebook 09 partition {chunk_id}: {input_path}. "
            "Rerun Notebook 09 Cell 4 if its restart files were removed."
        )
    actual_input_hash = (
        sha256_file(input_path) if VERIFY_INPUT_HASHES else expected_input_hash
    )
    if actual_input_hash != expected_input_hash:
        raise RuntimeError(
            f"Notebook 09 partition {chunk_id} hash mismatch: "
            f"{actual_input_hash} != {expected_input_hash}."
        )

    marker_valid = False
    marker: dict[str, Any] = {}
    if (
        marker_path.is_file()
        and output_path.is_file()
        and event_path.is_file()
        and validation_path.is_file()
    ):
        try:
            marker = load_json(marker_path)
            marker_valid = (
                marker.get("pipeline_version") == PIPELINE_VERSION
                and marker.get("basis_hash") == basis_hash
                and marker.get("input_sha256") == expected_input_hash
                and int(marker.get("rows", -1)) == int(manifest_row["rows"])
                and int(marker.get("occurrences", -1))
                == int(manifest_row["occurrences"])
                and marker.get("output_sha256") == sha256_file(output_path)
                and marker.get("event_sha256") == sha256_file(event_path)
                and marker.get("validation_sha256")
                == sha256_file(validation_path)
            )
        except Exception:
            marker_valid = False

    if marker_valid:
        print(
            f"Partition {manifest_index + 1:02d}/{len(notebook9_manifest):02d} "
            f"({chunk_id}): reused {int(marker['occurrences']):,} occurrences"
        )
        partition_event = pd.read_csv(event_path)
    else:
        partition_state_counts: Counter[str] = Counter()
        partition_loss_totals: Counter[str] = Counter()
        partition_minimum_raw: dict[str, float] = {}
        maximum_component_error = 0.0
        maximum_policy_error = 0.0
        maximum_uninsured_error = 0.0
        nonfinite_outputs = 0
        partition_rows = 0
        partition_occurrences: set[str] = set()
        partition_event_frames: list[pd.DataFrame] = []
        temporary = output_path.with_suffix(output_path.suffix + ".tmp")
        batch_rows = EVENT_BATCH_SIZE * EXPECTED_SITES
        first_batch = True

        with temporary.open("wb") as raw:
            with gzip.GzipFile(
                filename="",
                mode="wb",
                fileobj=raw,
                compresslevel=GZIP_COMPRESSLEVEL,
                mtime=0,
            ) as compressed:
                with io.TextIOWrapper(
                    compressed, encoding="utf-8", newline=""
                ) as text_output:
                    for ground_motion_batch in pd.read_csv(
                        input_path,
                        usecols=GROUND_MOTION_REQUIRED_COLUMNS,
                        chunksize=batch_rows,
                    ):
                        output_batch, diagnostics = build_paired_damage_loss_batch(
                            ground_motion_batch,
                            site_parameters,
                        )
                        output_batch.to_csv(
                            text_output,
                            index=False,
                            header=first_batch,
                            float_format="%.17g",
                            lineterminator="\n",
                        )
                        first_batch = False
                        partition_rows += len(output_batch)
                        partition_occurrences.update(
                            output_batch["occurrence_id"].astype(str).unique()
                        )
                        partition_event_frames.append(
                            summarize_occurrences(output_batch)
                        )
                        for prefix in CASE_PREFIXES.values():
                            for component in COMPONENT_NAMESPACES:
                                states = output_batch[
                                    f"{prefix}_{component}_damage_state"
                                ].to_numpy(dtype=int)
                                counts = np.bincount(states, minlength=5)
                                for state, count in enumerate(counts):
                                    partition_state_counts[
                                        f"{prefix}_{component}_{state}"
                                    ] += int(count)
                        for column in loss_total_columns:
                            partition_loss_totals[column] += float(
                                output_batch[column]
                                .to_numpy(dtype=np.float64)
                                .sum(dtype=np.float64)
                            )
                        for key, value in diagnostics.minimum_raw_probability.items():
                            partition_minimum_raw[key] = min(
                                partition_minimum_raw.get(key, math.inf),
                                float(value),
                            )
                        maximum_component_error = max(
                            maximum_component_error,
                            diagnostics.maximum_component_reconciliation_error_2022_usd,
                        )
                        maximum_policy_error = max(
                            maximum_policy_error,
                            diagnostics.maximum_policy_reconciliation_error_2022_usd,
                        )
                        maximum_uninsured_error = max(
                            maximum_uninsured_error,
                            diagnostics.maximum_uninsured_decomposition_error_2022_usd,
                        )
                        nonfinite_outputs += diagnostics.nonfinite_output_count
        temporary.replace(output_path)

        partition_event = pd.concat(
            partition_event_frames, ignore_index=True
        ).sort_values("occurrence_ordinal").reset_index(drop=True)
        write_gzip_csv_deterministic(partition_event, event_path)
        chunk_validation_rows: list[dict[str, object]] = []
        add_check(
            chunk_validation_rows,
            "partition_rows",
            partition_rows == int(manifest_row["rows"]),
            f"rows={partition_rows}; expected={int(manifest_row['rows'])}",
        )
        add_check(
            chunk_validation_rows,
            "partition_occurrences",
            len(partition_occurrences) == int(manifest_row["occurrences"]),
            (
                f"occurrences={len(partition_occurrences)}; "
                f"expected={int(manifest_row['occurrences'])}"
            ),
        )
        add_check(
            chunk_validation_rows,
            "event_summary_rows",
            len(partition_event) == len(partition_occurrences),
            f"event_rows={len(partition_event)}",
        )
        add_check(
            chunk_validation_rows,
            "component_loss_reconciliation",
            maximum_component_error <= ROW_TOLERANCE_USD,
            f"maximum_error={maximum_component_error:.6e}",
        )
        add_check(
            chunk_validation_rows,
            "policy_loss_reconciliation",
            maximum_policy_error <= ROW_TOLERANCE_USD,
            f"maximum_error={maximum_policy_error:.6e}",
        )
        add_check(
            chunk_validation_rows,
            "uninsured_decomposition",
            maximum_uninsured_error <= ROW_TOLERANCE_USD,
            f"maximum_error={maximum_uninsured_error:.6e}",
        )
        add_check(
            chunk_validation_rows,
            "all_numeric_outputs_finite",
            nonfinite_outputs == 0,
            f"nonfinite={nonfinite_outputs}",
        )
        chunk_validation = pd.DataFrame(chunk_validation_rows)
        write_csv(chunk_validation, validation_path)
        if not normalize_boolean_series(chunk_validation["passed"]).all():
            raise RuntimeError(
                f"Notebook 10 partition {chunk_id} failed validation."
            )

        marker = {
            "pipeline_version": PIPELINE_VERSION,
            "basis_hash": basis_hash,
            "chunk_id": chunk_id,
            "input_path": project_relative_path(input_path),
            "input_sha256": expected_input_hash,
            "rows": partition_rows,
            "occurrences": len(partition_occurrences),
            "state_counts": dict(partition_state_counts),
            "loss_totals_2022_usd": dict(partition_loss_totals),
            "minimum_raw_probability": partition_minimum_raw,
            "maximum_component_reconciliation_error_2022_usd": maximum_component_error,
            "maximum_policy_reconciliation_error_2022_usd": maximum_policy_error,
            "maximum_uninsured_decomposition_error_2022_usd": maximum_uninsured_error,
            "nonfinite_output_count": nonfinite_outputs,
            "output_path": project_relative_path(output_path),
            "output_sha256": sha256_file(output_path),
            "event_path": project_relative_path(event_path),
            "event_sha256": sha256_file(event_path),
            "validation_path": project_relative_path(validation_path),
            "validation_sha256": sha256_file(validation_path),
        }
        write_json(marker_path, marker)
        print(
            f"Partition {manifest_index + 1:02d}/{len(notebook9_manifest):02d} "
            f"({chunk_id}): {len(partition_occurrences):,} occurrences complete"
        )

    output_chunk_paths.append(output_path)
    event_summary_frames.append(partition_event)
    public_manifest_rows.append(
        {
            "chunk_number": int(manifest_row["chunk_number"]),
            "chunk_id": chunk_id,
            "occurrences": int(marker["occurrences"]),
            "rows": int(marker["rows"]),
            "input_path": project_relative_path(input_path),
            "input_sha256": str(marker["input_sha256"]),
            "output_path": project_relative_path(output_path),
            "output_sha256": str(marker["output_sha256"]),
            "output_bytes": int(output_path.stat().st_size),
            "event_path": project_relative_path(event_path),
            "event_sha256": str(marker["event_sha256"]),
            "marker_path": project_relative_path(marker_path),
            "marker_sha256": sha256_file(marker_path),
            "validation_path": project_relative_path(validation_path),
            "validation_sha256": str(marker["validation_sha256"]),
            "status": "complete",
        }
    )

chunk_manifest = pd.DataFrame(public_manifest_rows).sort_values(
    "chunk_number"
).reset_index(drop=True)
write_csv(chunk_manifest, CHUNK_MANIFEST_PATH)
combine_gzip_csv_files(output_chunk_paths, FINAL_BUILDING_LOSS_PATH)
event_summary = pd.concat(event_summary_frames, ignore_index=True)
event_summary = event_summary.sort_values("occurrence_ordinal").reset_index(
    drop=True
)
write_gzip_csv_deterministic(event_summary, FINAL_EVENT_LOSS_PATH)

print("=" * 78)
print("NOTEBOOK 10 CELL 3: ALL PAIRED DAMAGE-LOSS PARTITIONS COMPLETE")
print("=" * 78)
print(f"Partitions:                   {len(chunk_manifest):,}")
print(f"Occurrences:                  {len(event_summary):,}")
print(f"Rows:                         {int(chunk_manifest['rows'].sum()):,}")
print(f"Building output:              {project_relative_path(FINAL_BUILDING_LOSS_PATH)}")
print("Next: construct complete annual loss series and validate I0 totals.")

In [ ]:
event_summary = pd.read_csv(FINAL_EVENT_LOSS_PATH)
chunk_manifest = pd.read_csv(CHUNK_MANIFEST_PATH)
production_state_counts: Counter[str] = Counter()
production_loss_totals: Counter[str] = Counter()
production_maximum_errors = {
    "component": 0.0,
    "policy": 0.0,
    "uninsured": 0.0,
}
production_minimum_raw: dict[str, float] = {}
for marker_value in chunk_manifest["marker_path"]:
    marker = load_json(resolve_repository_path(marker_value))
    production_state_counts.update(
        {key: int(value) for key, value in marker["state_counts"].items()}
    )
    for key, value in marker["loss_totals_2022_usd"].items():
        production_loss_totals[key] += float(value)
    production_maximum_errors["component"] = max(
        production_maximum_errors["component"],
        float(marker["maximum_component_reconciliation_error_2022_usd"]),
    )
    production_maximum_errors["policy"] = max(
        production_maximum_errors["policy"],
        float(marker["maximum_policy_reconciliation_error_2022_usd"]),
    )
    production_maximum_errors["uninsured"] = max(
        production_maximum_errors["uninsured"],
        float(marker["maximum_uninsured_decomposition_error_2022_usd"]),
    )
    for key, value in marker["minimum_raw_probability"].items():
        production_minimum_raw[key] = min(
            production_minimum_raw.get(key, math.inf), float(value)
        )

years = np.arange(1, EXPECTED_CATALOG_YEARS + 1, dtype=np.int32)
annual = pd.DataFrame({"catalog_year": years})
occurrence_counts = event_summary.groupby("catalog_year").size()
annual_occurrence_count = np.zeros(EXPECTED_CATALOG_YEARS, dtype=np.int16)
annual_occurrence_count[
    occurrence_counts.index.to_numpy(dtype=int) - 1
] = occurrence_counts.to_numpy(dtype=np.int16)
annual["catalog_occurrence_count"] = annual_occurrence_count

loss_basis_suffixes = {
    "ground_up": "total_ground_up_loss_2022_usd",
    "gross_insured": "gross_insured_loss_2022_usd",
    "uninsured": "uninsured_loss_2022_usd",
}
for case_name, prefix in CASE_PREFIXES.items():
    columns = [
        f"{prefix}_{suffix}" for suffix in loss_basis_suffixes.values()
    ]
    grouped = event_summary.groupby("catalog_year", sort=True)[columns]
    annual_sums = grouped.sum()
    annual_maxima = grouped.max()
    row_indices = annual_sums.index.to_numpy(dtype=int) - 1
    for loss_basis, suffix in loss_basis_suffixes.items():
        source_column = f"{prefix}_{suffix}"
        aep_values = np.zeros(EXPECTED_CATALOG_YEARS, dtype=np.float64)
        oep_values = np.zeros(EXPECTED_CATALOG_YEARS, dtype=np.float64)
        aep_values[row_indices] = annual_sums[source_column].to_numpy(
            dtype=np.float64
        )
        oep_values[row_indices] = annual_maxima[source_column].to_numpy(
            dtype=np.float64
        )
        annual[f"{prefix}_{loss_basis}_aep_2022_usd"] = aep_values
        annual[f"{prefix}_{loss_basis}_oep_2022_usd"] = oep_values
write_gzip_csv_deterministic(annual, FINAL_ANNUAL_LOSS_PATH)

pml_rows: list[dict[str, object]] = []
for case_name, prefix in CASE_PREFIXES.items():
    for loss_basis in loss_basis_suffixes:
        for curve_type in ("aep", "oep"):
            column = f"{prefix}_{loss_basis}_{curve_type}_2022_usd"
            values = annual[column].to_numpy(dtype=np.float64)
            for return_period in RETURN_PERIODS:
                pml, rank = empirical_pml(values, return_period)
                pml_rows.append(
                    {
                        "case_name": case_name,
                        "case_prefix": prefix,
                        "loss_basis": loss_basis,
                        "curve_type": curve_type,
                        "return_period_years": return_period,
                        "order_statistic_rank": rank,
                        "tail_support_sufficient": rank >= 20,
                        "pml_2022_usd": pml,
                    }
                )
pml_table = pd.DataFrame(pml_rows)
write_csv(pml_table, PML_TABLE_PATH)

def pml_lookup(prefix: str, return_period: int) -> float:
    row = pml_table.loc[
        pml_table["case_prefix"].eq(prefix)
        & pml_table["loss_basis"].eq("gross_insured")
        & pml_table["curve_type"].eq("oep")
        & pml_table["return_period_years"].eq(return_period),
        "pml_2022_usd",
    ]
    if len(row) != 1:
        raise RuntimeError("Gross-insured OEP lookup is not unique.")
    return float(row.iloc[0])

case_rows: list[dict[str, object]] = []
for case_name, prefix in CASE_PREFIXES.items():
    row: dict[str, object] = {
        "case_name": case_name,
        "case_prefix": prefix,
        "catalog_years": EXPECTED_CATALOG_YEARS,
        "occurrences": EXPECTED_OCCURRENCES,
        "portfolio_replacement_value_2022_usd": replacement_total,
    }
    for loss_basis, suffix in loss_basis_suffixes.items():
        total = float(production_loss_totals[f"{prefix}_{suffix}"])
        aep = annual[f"{prefix}_{loss_basis}_aep_2022_usd"].to_numpy(
            dtype=np.float64
        )
        oep = annual[f"{prefix}_{loss_basis}_oep_2022_usd"].to_numpy(
            dtype=np.float64
        )
        row[f"{loss_basis}_loss_total_2022_usd"] = total
        row[f"{loss_basis}_aal_2022_usd"] = total / EXPECTED_CATALOG_YEARS
        row[f"{loss_basis}_annual_standard_deviation_2022_usd"] = float(
            np.std(aep, ddof=0)
        )
        row[f"maximum_{loss_basis}_aep_2022_usd"] = float(aep.max())
        row[f"maximum_{loss_basis}_oep_2022_usd"] = float(oep.max())
        row[f"positive_{loss_basis}_aep_years"] = int(np.count_nonzero(aep))
        row[f"positive_{loss_basis}_oep_years"] = int(np.count_nonzero(oep))
    row["insurance_recovery_share_of_ground_up"] = (
        float(row["gross_insured_loss_total_2022_usd"])
        / float(row["ground_up_loss_total_2022_usd"])
    )
    row["gross_insured_250yr_oep_pml_2022_usd"] = pml_lookup(prefix, 250)
    row["gross_insured_500yr_oep_pml_2022_usd"] = pml_lookup(prefix, 500)
    row["gross_insured_1000yr_oep_pml_2022_usd"] = pml_lookup(prefix, 1_000)
    case_rows.append(row)
case_summary = pd.DataFrame(case_rows)
i0_row = case_summary.loc[case_summary["case_prefix"].eq("i0")].iloc[0]
for metric in [
    "ground_up_aal_2022_usd",
    "gross_insured_aal_2022_usd",
    "gross_insured_250yr_oep_pml_2022_usd",
    "gross_insured_500yr_oep_pml_2022_usd",
    "gross_insured_1000yr_oep_pml_2022_usd",
]:
    baseline = float(i0_row[metric])
    case_summary[f"{metric.removesuffix('_2022_usd')}_change_vs_i0_percent"] = np.where(
        baseline != 0.0,
        100.0 * (case_summary[metric].to_numpy(dtype=float) / baseline - 1.0),
        0.0,
    )
write_csv(case_summary, CASE_SUMMARY_PATH)

damage_rows: list[dict[str, object]] = []
for case_name, prefix in CASE_PREFIXES.items():
    for component in COMPONENT_NAMESPACES:
        for state, state_name in enumerate(DAMAGE_STATES):
            count = int(production_state_counts[f"{prefix}_{component}_{state}"])
            damage_rows.append(
                {
                    "case_name": case_name,
                    "case_prefix": prefix,
                    "component": component,
                    "damage_state": state,
                    "damage_state_name": state_name,
                    "rows": count,
                    "proportion": count / EXPECTED_ROWS,
                }
            )
damage_state_summary = pd.DataFrame(damage_rows)
write_csv(damage_state_summary, DAMAGE_STATE_SUMMARY_PATH)

final_validation_rows: list[dict[str, object]] = []
input_checks = pd.read_csv(INPUT_VALIDATION_PATH)
control_checks = pd.read_csv(CONTROL_VALIDATION_PATH)
add_check(
    final_validation_rows,
    "input_and_control_validation_passed",
    bool(
        normalize_boolean_series(input_checks["passed"]).all()
        and normalize_boolean_series(control_checks["passed"]).all()
    ),
    f"input_checks={len(input_checks)}; control_checks={len(control_checks)}",
)
add_check(
    final_validation_rows,
    "full_catalog_dimensions",
    int(chunk_manifest["rows"].sum()) == EXPECTED_ROWS
    and len(event_summary) == EXPECTED_OCCURRENCES
    and event_summary["occurrence_id"].nunique() == EXPECTED_OCCURRENCES
    and event_summary["sites"].eq(EXPECTED_SITES).all(),
    (
        f"rows={int(chunk_manifest['rows'].sum())}; events={len(event_summary)}; "
        f"site_range={event_summary['sites'].min()}-{event_summary['sites'].max()}"
    ),
)
building_header = pd.read_csv(FINAL_BUILDING_LOSS_PATH, nrows=0)
add_check(
    final_validation_rows,
    "building_output_schema",
    list(building_header.columns) == PAIRED_LOSS_OUTPUT_COLUMNS,
    f"columns={len(building_header.columns)}",
)
state_mismatches = {
    key: int(production_state_counts[key]) - expected
    for key, expected in EXPECTED_PHASE1_STATE_COUNTS.items()
    if int(production_state_counts[key]) != expected
}
add_check(
    final_validation_rows,
    "i0_damage_state_counts_reproduce_phase1_exactly",
    not state_mismatches,
    f"mismatches={state_mismatches}",
)
phase1_total_errors = {
    key: float(production_loss_totals[key]) - expected
    for key, expected in EXPECTED_PHASE1_TOTALS.items()
}
add_check(
    final_validation_rows,
    "i0_financial_totals_reproduce_phase1",
    max(abs(value) for value in phase1_total_errors.values())
    <= AGGREGATE_TOLERANCE_USD,
    f"errors_2022_usd={phase1_total_errors}",
)
add_check(
    final_validation_rows,
    "row_level_component_and_policy_reconciliation",
    max(production_maximum_errors.values()) <= ROW_TOLERANCE_USD,
    f"maximum_errors_2022_usd={production_maximum_errors}",
)
for prefix in CASE_PREFIXES.values():
    component_total = sum(
        float(
            production_loss_totals[
                f"{prefix}_{component}_ground_up_loss_2022_usd"
            ]
        )
        for component in COMPONENT_NAMESPACES
    )
    ground_up_total = float(
        production_loss_totals[f"{prefix}_total_ground_up_loss_2022_usd"]
    )
    gross_total = float(
        production_loss_totals[f"{prefix}_gross_insured_loss_2022_usd"]
    )
    uninsured_total = float(
        production_loss_totals[f"{prefix}_uninsured_loss_2022_usd"]
    )
    add_check(
        final_validation_rows,
        f"{prefix}_aggregate_loss_reconciliation",
        abs(component_total - ground_up_total) <= AGGREGATE_TOLERANCE_USD
        and abs(ground_up_total - gross_total - uninsured_total)
        <= AGGREGATE_TOLERANCE_USD,
        (
            f"component_error={component_total-ground_up_total:.6e}; "
            f"policy_error={ground_up_total-gross_total-uninsured_total:.6e}"
        ),
    )
    limit_absorbed = float(
        production_loss_totals[
            f"{prefix}_policy_limit_absorbed_loss_2022_usd"
        ]
    )
    coinsurance_absorbed = float(
        production_loss_totals[
            f"{prefix}_coinsurance_absorbed_loss_2022_usd"
        ]
    )
    add_check(
        final_validation_rows,
        f"{prefix}_baseline_limit_and_coinsurance_nonbinding",
        abs(limit_absorbed) <= AGGREGATE_TOLERANCE_USD
        and abs(coinsurance_absorbed) <= AGGREGATE_TOLERANCE_USD,
        (
            f"limit_absorbed={limit_absorbed:.6f}; "
            f"coinsurance_absorbed={coinsurance_absorbed:.6f}"
        ),
    )
add_check(
    final_validation_rows,
    "annual_series_includes_all_zero_event_years",
    len(annual) == EXPECTED_CATALOG_YEARS
    and int(np.count_nonzero(annual_occurrence_count))
    == EXPECTED_OCCUPIED_CATALOG_YEARS
    and int(np.count_nonzero(annual_occurrence_count == 0))
    == EXPECTED_ZERO_EVENT_YEARS,
    (
        f"years={len(annual)}; occupied={int(np.count_nonzero(annual_occurrence_count))}; "
        f"zero_event={int(np.count_nonzero(annual_occurrence_count == 0))}"
    ),
)
phase1_metric_errors = {
    metric: float(i0_row[metric]) - float(expected)
    for metric, expected in EXPECTED_PHASE1_ANNUAL_METRICS.items()
    if not metric.endswith("_years")
}
phase1_count_mismatches = {
    metric: int(i0_row[metric]) - int(expected)
    for metric, expected in EXPECTED_PHASE1_ANNUAL_METRICS.items()
    if metric.endswith("_years") and int(i0_row[metric]) != int(expected)
}
add_check(
    final_validation_rows,
    "i0_annual_metrics_reproduce_phase1",
    max(abs(value) for value in phase1_metric_errors.values())
    <= AGGREGATE_TOLERANCE_USD
    and not phase1_count_mismatches,
    (
        "maximum_numeric_error_2022_usd="
        f"{max(abs(value) for value in phase1_metric_errors.values()):.6e}; "
        f"count_mismatches={phase1_count_mismatches}"
    ),
)
event_total_errors: dict[str, float] = {}
annual_total_errors: dict[str, float] = {}
for prefix in CASE_PREFIXES.values():
    for loss_basis, suffix in loss_basis_suffixes.items():
        column = f"{prefix}_{suffix}"
        event_total_errors[column] = (
            float(
                event_summary[column]
                .to_numpy(dtype=np.float64)
                .sum(dtype=np.float64)
            )
            - float(production_loss_totals[column])
        )
        annual_total_errors[column] = (
            float(
                annual[f"{prefix}_{loss_basis}_aep_2022_usd"]
                .to_numpy(dtype=np.float64)
                .sum(dtype=np.float64)
            )
            - float(production_loss_totals[column])
        )
maximum_event_error = max(
    abs(value) for value in event_total_errors.values()
)
maximum_annual_error = max(
    abs(value) for value in annual_total_errors.values()
)
add_check(
    final_validation_rows,
    "building_event_annual_totals_reconcile",
    max(maximum_event_error, maximum_annual_error)
    <= AGGREGATE_TOLERANCE_USD,
    (
        f"maximum_event_error={maximum_event_error:.6e}; "
        f"maximum_annual_error={maximum_annual_error:.6e}"
    ),
)
pml_monotonic = True
for _, group in pml_table.groupby(
    ["case_prefix", "loss_basis", "curve_type"], sort=False
):
    ordered = group.sort_values("return_period_years")[
        "pml_2022_usd"
    ].to_numpy(dtype=float)
    pml_monotonic &= bool(np.all(np.diff(ordered) >= -ROW_TOLERANCE_USD))
add_check(
    final_validation_rows,
    "pml_tables_nondecreasing_by_return_period",
    pml_monotonic,
    f"rows={len(pml_table)}",
)
add_check(
    final_validation_rows,
    "public_paths_are_repository_relative",
    bool(
        chunk_manifest[
            [
                "input_path",
                "output_path",
                "event_path",
                "marker_path",
                "validation_path",
            ]
        ]
        .astype(str)
        .apply(lambda column: ~column.str.contains(r"^[A-Za-z]:|\\", regex=True))
        .all()
        .all()
    ),
    "Notebook 10 public metadata contains only repository-relative paths.",
)
add_check(
    final_validation_rows,
    "extreme_return_period_tail_support_documented",
    False,
    (
        "Return periods with order-statistic rank below 20 are diagnostics, "
        "not headline estimates."
    ),
    severity="warning",
)

final_validation = pd.DataFrame(final_validation_rows)
write_csv(final_validation, FINAL_VALIDATION_PATH)
critical_failures = final_validation.loc[
    final_validation["severity"].eq("critical")
    & ~normalize_boolean_series(final_validation["passed"])
]
if not critical_failures.empty:
    display(critical_failures)
    raise RuntimeError("Notebook 10 full-catalog validation failed.")

display(
    case_summary[
        [
            "case_name",
            "ground_up_aal_2022_usd",
            "gross_insured_aal_2022_usd",
            "gross_insured_250yr_oep_pml_2022_usd",
            "gross_insured_500yr_oep_pml_2022_usd",
            "gross_insured_1000yr_oep_pml_2022_usd",
        ]
    ]
)
print("=" * 78)
print("NOTEBOOK 10 CELL 4: ANNUAL LOSS DISTRIBUTIONS VALIDATED")
print("=" * 78)
print(f"Annual rows:                  {len(annual):,}")
print(f"Zero-event years:             {int(np.count_nonzero(annual_occurrence_count == 0)):,}")
print(f"PML rows:                     {len(pml_table):,}")
print(f"Critical failures:            {len(critical_failures):,}")
print(f"Paired loss SHA-256:          {sha256_file(FINAL_BUILDING_LOSS_PATH)}")
print("Next: write the portable Notebook 11 handoff.")

In [ ]:
final_validation = pd.read_csv(FINAL_VALIDATION_PATH)
critical_failures = final_validation.loc[
    final_validation["severity"].astype(str).str.lower().eq("critical")
    & ~normalize_boolean_series(final_validation["passed"])
]
if not critical_failures.empty:
    raise RuntimeError(
        "Notebook 10 handoff is blocked by failed critical validation."
    )

artifact_paths = [
    INPUT_VALIDATION_PATH,
    CONTROL_VALIDATION_PATH,
    MODEL_SPECIFICATION_PATH,
    CHUNK_MANIFEST_PATH,
    CASE_SUMMARY_PATH,
    DAMAGE_STATE_SUMMARY_PATH,
    PML_TABLE_PATH,
    FINAL_VALIDATION_PATH,
    FINAL_BUILDING_LOSS_PATH,
    FINAL_EVENT_LOSS_PATH,
    FINAL_ANNUAL_LOSS_PATH,
]
artifact_inventory = [
    {
        "path": project_relative_path(path),
        "bytes": int(path.stat().st_size),
        "sha256": sha256_file(path),
    }
    for path in artifact_paths
]
handoff = {
    "schema_version": SCHEMA_VERSION,
    "pipeline_version": PIPELINE_VERSION,
    "notebook10_complete": True,
    "dependence_cases": list(CASE_PREFIXES),
    "frozen_controls": {
        "phase1_release": "v1.0.0",
        "phase1_commit": "be93474ce2ab78d8002d49ae861adb641ae2741d",
        "catalog_years": EXPECTED_CATALOG_YEARS,
        "occurrences": EXPECTED_OCCURRENCES,
        "sites": EXPECTED_SITES,
        "paired_ground_motion_sha256": EXPECTED_HASHES[
            "paired_ground_motion"
        ],
        "damage_stream_namespaces": COMPONENT_NAMESPACES,
        "portfolio_replacement_value_2022_usd": replacement_total,
    },
    "damage_model": {
        "states": list(DAMAGE_STATES),
        "components": list(COMPONENT_NAMESPACES),
        "production_im": "SA(0.4 s) in g",
        "equation": "P(DS>=k|Sa)=Phi((ln(Sa)-ln(theta_k))/beta_k)",
        "paired_uniform_rule": (
            "The same component-specific occurrence-site uniform is reused "
            "across I0, C1, and C2."
        ),
    },
    "policy": {
        "scenario_id": "baseline_full_coverage_10pct_deductible_v1",
        "coverage_share": 1.0,
        "deductible_share_of_replacement": 0.1,
        "limit_share_of_replacement": 1.0,
        "coinsurance_share": 1.0,
        "application": "per building per occurrence",
    },
    "catalog": {
        "rows": EXPECTED_ROWS,
        "annual_rows": EXPECTED_CATALOG_YEARS,
        "occupied_years": EXPECTED_OCCUPIED_CATALOG_YEARS,
        "zero_event_years": EXPECTED_ZERO_EVENT_YEARS,
    },
    "case_results": json.loads(
        case_summary.to_json(orient="records", double_precision=15)
    ),
    "output_contract": {
        "building_loss": {
            "path": project_relative_path(FINAL_BUILDING_LOSS_PATH),
            "rows": EXPECTED_ROWS,
            "row_granularity": "one catalog occurrence and one building",
        },
        "event_loss": {
            "path": project_relative_path(FINAL_EVENT_LOSS_PATH),
            "rows": EXPECTED_OCCURRENCES,
            "row_granularity": "one catalog occurrence",
        },
        "annual_loss": {
            "path": project_relative_path(FINAL_ANNUAL_LOSS_PATH),
            "rows": EXPECTED_CATALOG_YEARS,
            "row_granularity": "one catalog year including zero-event years",
        },
        "pml_table": {
            "path": project_relative_path(PML_TABLE_PATH),
            "rows": len(pml_table),
        },
    },
    "validation": {
        "path": project_relative_path(FINAL_VALIDATION_PATH),
        "sha256": sha256_file(FINAL_VALIDATION_PATH),
        "checks": len(final_validation),
        "critical_failures": 0,
        "warnings": int(
            (
                final_validation["severity"].eq("warning")
                & ~normalize_boolean_series(final_validation["passed"])
            ).sum()
        ),
        "i0_damage_state_counts_exact": True,
        "i0_financial_total_tolerance_2022_usd": AGGREGATE_TOLERANCE_USD,
        "row_reconciliation_tolerance_2022_usd": ROW_TOLERANCE_USD,
    },
    "artifact_inventory": artifact_inventory,
    "next_notebook": "11_reinsurance_sensitivity_and_capital.ipynb",
    "next_task": (
        "Apply the frozen occurrence XoL and occurrence-plus-aggregate "
        "reinsurance designs to all three gross-insured event and annual "
        "loss distributions, then quantify retained, ceded, capital, TVaR, "
        "diversification, and risk-adjusted-return effects."
    ),
}
write_json(FINAL_HANDOFF_PATH, handoff)

print("=" * 78)
print("NOTEBOOK 10 COMPLETE: CORRELATED DAMAGE AND POLICY LOSS VALIDATED")
print("=" * 78)
print(f"Dependence cases:             {len(CASE_PREFIXES):,}")
print(f"Catalog occurrences:          {EXPECTED_OCCURRENCES:,}")
print(f"Occurrence-building rows:     {EXPECTED_ROWS:,}")
print(f"Annual rows including zeros:  {EXPECTED_CATALOG_YEARS:,}")
print(f"Validation checks:            {len(final_validation):,}")
print(f"Critical failures:            {len(critical_failures):,}")
print(f"Paired building loss:         {project_relative_path(FINAL_BUILDING_LOSS_PATH)}")
print(f"Final handoff:                {project_relative_path(FINAL_HANDOFF_PATH)}")
print("Next: Notebook 11 reinsurance sensitivity and capital.")

## Interpretation boundary

Notebook 10 establishes how spatial dependence changes the gross
damage and insured-loss distributions while preserving the Phase 1
model controls. AAL differences are reported, but the main decision
signal is expected in the occurrence and annual tails.

This notebook does not yet select a reinsurance attachment or limit,
calculate ceded and retained TVaR, or report risk-adjusted return.
Those decisions require applying identical treaty structures to the
three validated loss distributions in Notebook 11. The final Phase 2
conclusion must quantify those financial changes and must not stop at
a generic statement that correlation increases tail loss.